In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import numpy as np
import itertools


In [3]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [4]:
# ------------------------------
# Matrix Factorization Model
# ------------------------------
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

        nn.init.normal_(self.user_factors.weight, std=0.1)
        nn.init.normal_(self.item_factors.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        dot = (self.user_factors(users) * self.item_factors(items)).sum(dim=1, keepdim=True)
        preds = dot + self.user_bias(users) + self.item_bias(items) + self.global_bias
        return preds.squeeze()

In [5]:
# ------------------------------
# Dataset Class
# ------------------------------
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_id"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


In [6]:
# ------------------------------
# Training and Evaluation
# ------------------------------
def train_model(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(ratings)
    return np.sqrt(total_loss / len(train_loader.dataset))

def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for users, items, ratings in loader:
            users, items, ratings = users.to(device), items.to(device), ratings.to(device)
            preds = model(users, items)
            loss = criterion(preds, ratings)
            total_loss += loss.item() * len(ratings)
    return np.sqrt(total_loss / len(loader.dataset))


In [7]:
df = pd.read_csv("dataset/u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
df.drop(columns=["timestamp"], inplace=True)
df["user_id"] -= df["user_id"].min()
df["item_id"] -= df["item_id"].min()

n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()

train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.MSELoss()

param_grid = {
    "n_factors": [5,10, 16, 32, 64],
    "lr": [ 1e-3, 1e-2, 1e-1],
    "weight_decay": [0, 1e-6, 1e-5, 1e-4]
}

best_val_rmse = float("inf")
best_params = {}
best_model_state = None



In [8]:
for n_factors, lr, weight_decay in itertools.product(
    param_grid["n_factors"], param_grid["lr"], param_grid["weight_decay"]
):
    print(f"Testing n_factors={n_factors}, lr={lr}, weight_decay={weight_decay}")
    model = MatrixFactorization(n_users, n_items, n_factors).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
    val_loader = DataLoader(RatingsDataset(val_df), batch_size=1024)

    for epoch in range(10):  # fewer epochs for tuning speed
        train_rmse = train_model(model, train_loader, optimizer, criterion, device)

    val_rmse = evaluate_model(model, val_loader, criterion, device)
    print(f"  => Val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_params = {
            "n_factors": n_factors,
            "lr": lr,
            "weight_decay": weight_decay
        }
        best_model_state = model.state_dict()

print("\nBest Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"✅ Best Validation RMSE: {best_val_rmse:.4f}")



Testing n_factors=5, lr=0.001, weight_decay=0
  => Val RMSE: 1.6776
Testing n_factors=5, lr=0.001, weight_decay=1e-06
  => Val RMSE: 1.8848
Testing n_factors=5, lr=0.001, weight_decay=1e-05
  => Val RMSE: 1.6768
Testing n_factors=5, lr=0.001, weight_decay=0.0001
  => Val RMSE: 1.6075
Testing n_factors=5, lr=0.01, weight_decay=0
  => Val RMSE: 0.9361
Testing n_factors=5, lr=0.01, weight_decay=1e-06
  => Val RMSE: 0.9347
Testing n_factors=5, lr=0.01, weight_decay=1e-05
  => Val RMSE: 0.9419
Testing n_factors=5, lr=0.01, weight_decay=0.0001
  => Val RMSE: 0.9182
Testing n_factors=5, lr=0.1, weight_decay=0
  => Val RMSE: 1.0202
Testing n_factors=5, lr=0.1, weight_decay=1e-06
  => Val RMSE: 1.0182
Testing n_factors=5, lr=0.1, weight_decay=1e-05
  => Val RMSE: 0.9993
Testing n_factors=5, lr=0.1, weight_decay=0.0001
  => Val RMSE: 0.9690
Testing n_factors=10, lr=0.001, weight_decay=0
  => Val RMSE: 1.3866
Testing n_factors=10, lr=0.001, weight_decay=1e-06
  => Val RMSE: 1.4040
Testing n_facto

In [9]:
# Final test evaluation
test_loader = DataLoader(RatingsDataset(test_df), batch_size=1024)
final_model = MatrixFactorization(n_users, n_items, best_params["n_factors"]).to(device)
final_model.load_state_dict(best_model_state)
test_rmse = evaluate_model(final_model, test_loader, criterion, device)
print(f"✅ Test RMSE with best model: {test_rmse:.4f}")


✅ Test RMSE with best model: 0.9232
